# Azure Container Apps Deployment

This notebook deploys the MLOps FastAPI app to Azure Container Apps via the Python SDK.

**Workflow:**
1. Create Azure Container Registry (ACR)
2. Build and push Docker image
3. Deploy to Azure Container Apps
4. Verify endpoints

> See `../container_registry_aci/` for the legacy ACI-based version.

In [ ]:
!pip install azure-identity azure-mgmt-resource azure-mgmt-containerregistry azure-mgmt-appcontainers python-dotenv requests

In [ ]:
import os
import subprocess
import requests as http_requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.containerregistry import ContainerRegistryManagementClient
from azure.mgmt.appcontainers import ContainerAppsAPIClient

load_dotenv()

SUBSCRIPTION_ID = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP = os.environ.get('AZURE_RESOURCE_GROUP', 'mlops-rg')
LOCATION = os.environ.get('AZURE_LOCATION', 'eastus')
ACR_NAME = os.environ.get('AZURE_ACR_NAME', 'mlopsacr')
APP_NAME = 'mlops-api'
ENV_NAME = 'mlops-env'
STORAGE_CONN_STR = os.environ['AZURE_STORAGE_CONNECTION_STRING']

credential = DefaultAzureCredential()
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
acr_client = ContainerRegistryManagementClient(credential, SUBSCRIPTION_ID)
aca_client = ContainerAppsAPIClient(credential, SUBSCRIPTION_ID)

IMAGE_URI = f'{ACR_NAME}.azurecr.io/{APP_NAME}:latest'

print(f'Subscription:   {SUBSCRIPTION_ID}')
print(f'Resource Group: {RESOURCE_GROUP}')
print(f'ACR:            {ACR_NAME}')
print(f'Image URI:      {IMAGE_URI}')

In [ ]:
# Create ACR (idempotent)
from azure.mgmt.containerregistry.models import Registry, Sku

print(f'Creating/updating ACR {ACR_NAME!r}...')
op = acr_client.registries.begin_create(
    RESOURCE_GROUP,
    ACR_NAME,
    Registry(
        location=LOCATION,
        sku=Sku(name='Basic'),
        admin_user_enabled=True,
    )
)
acr = op.result()
print(f'ACR login server: {acr.login_server}')

## Build and Push Docker Image

Run these commands in your terminal (Docker must be running locally):

In [ ]:
print('# Step 1: Login to ACR')
print(f'az acr login --name {ACR_NAME}')
print()
print('# Step 2: Build image')
print(f'docker build -t {APP_NAME} ../06-dockerize-app-and-fastapi/')
print()
print('# Step 3: Tag')
print(f'docker tag {APP_NAME} {IMAGE_URI}')
print()
print('# Step 4: Push')
print(f'docker push {IMAGE_URI}')

In [ ]:
# Get ACR credentials for Container Apps
creds = acr_client.registries.list_credentials(RESOURCE_GROUP, ACR_NAME)
ACR_USERNAME = creds.username
ACR_PASSWORD = creds.passwords[0].value
print(f'ACR credentials retrieved for: {ACR_NAME}')

In [ ]:
# Create Container Apps Managed Environment
from azure.mgmt.appcontainers.models import (
    ManagedEnvironment,
    ContainerApp,
    Configuration,
    Ingress,
    RegistryCredentials,
    Template,
    Container,
    EnvironmentVar,
    Secret,
)

print(f'Creating Container Apps environment {ENV_NAME!r}...')
env_op = aca_client.managed_environments.begin_create_or_update(
    RESOURCE_GROUP,
    ENV_NAME,
    ManagedEnvironment(location=LOCATION),
)
env = env_op.result()
print(f'Environment created: {env.name}')

In [ ]:
# Deploy Container App
print(f'Deploying Container App {APP_NAME!r}...')

app_op = aca_client.container_apps.begin_create_or_update(
    RESOURCE_GROUP,
    APP_NAME,
    ContainerApp(
        location=LOCATION,
        managed_environment_id=env.id,
        configuration=Configuration(
            ingress=Ingress(
                external=True,
                target_port=5000,
            ),
            registries=[
                RegistryCredentials(
                    server=f'{ACR_NAME}.azurecr.io',
                    username=ACR_USERNAME,
                    password_secret_ref='acr-password',
                )
            ],
            secrets=[
                Secret(name='acr-password', value=ACR_PASSWORD),
                Secret(name='storage-connection', value=STORAGE_CONN_STR),
            ],
        ),
        template=Template(
            containers=[
                Container(
                    name=APP_NAME,
                    image=IMAGE_URI,
                    env=[
                        EnvironmentVar(
                            name='AZURE_STORAGE_CONNECTION_STRING',
                            secret_ref='storage-connection',
                        ),
                    ],
                    resources={'cpu': 1.0, 'memory': '2Gi'},
                )
            ]
        ),
    ),
)
app = app_op.result()
APP_URL = f'https://{app.configuration.ingress.fqdn}'
print(f'Container App deployed: {APP_URL}')

In [ ]:
# Verify health endpoint
import time
time.sleep(10)  # wait for app to become ready

resp = http_requests.get(APP_URL, timeout=30)
print(f'Health check: {resp.status_code}')
print(resp.json())

In [ ]:
# Test pose classifier
payload = {'url': ['https://images.pexels.com/photos/1755385/pexels-photo-1755385.jpeg']}
resp = http_requests.post(f'{APP_URL}/api/v1/pose_classifier', json=payload, timeout=60)
print(f'Pose classifier: {resp.status_code}')
print(resp.json())

In [ ]:
# Teardown — delete entire resource group (removes ALL resources atomically)
# Uncomment to execute:
# op = resource_client.resource_groups.begin_delete(RESOURCE_GROUP)
# op.result()
# print(f'Resource group {RESOURCE_GROUP!r} deleted.')
print('Teardown skipped. Uncomment above to delete the entire resource group.')